In [6]:
# Cell 0: Config (replace ALPHA=0.80 with nothing — computed per country below)
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
from scipy.stats import norm
import scipy.stats as stats
import numpy as np

PATHS = {
    'M0': '../output/results/M0_results_5YCDS.csv',
    'M1': '../output/results/M1_results_5YCDS.csv',
    'M2': '../output/results/M2_results_5YCDS.csv',
}
DATE_COL    = 'date'
COUNTRY_COL = 'country'
DD_COL      = 'distance_to_distress'
CDS_COL     = 'cds_spread'
HORIZONS    = [1, 2, 4, 8]
EXPORTERS   = ['Saudi Arabia', 'Abu Dhabi', 'Qatar', 'Colombia',
               'Mexico', 'Brazil', 'Egypt', 'Malaysia']
CONTROLS    = ['Chile', 'China', 'Indonesia', 'Philippines', 'South Africa',
               'South Korea', 'Thailand', 'Turkey']
RECOVERY = 0.40
HORIZON  = 5

In [7]:
# Cell 1: Build df with observed CDS and model DDs
df_obs = pd.read_csv(PATHS['M0'], parse_dates=[DATE_COL],
                     usecols=[DATE_COL, COUNTRY_COL, CDS_COL])

dd_frames = [df_obs.set_index([DATE_COL, COUNTRY_COL])]

for model_name, path in PATHS.items():
    df = pd.read_csv(path, parse_dates=[DATE_COL],
                     usecols=[DATE_COL, COUNTRY_COL, DD_COL])
    df = df.rename(columns={DD_COL: f'DD_{model_name}'})
    dd_frames.append(df.set_index([DATE_COL, COUNTRY_COL]))

panel = pd.concat(dd_frames, axis=1).reset_index()
panel = panel[panel[COUNTRY_COL].isin(EXPORTERS + CONTROLS)].dropna()
panel['group']    = panel[COUNTRY_COL].apply(
    lambda x: 'Exporter' if x in EXPORTERS else 'Control'
)
panel['exporter'] = (panel['group'] == 'Exporter').astype(float)

panel

,date,country,cds_spread,DD_M0,DD_M1,DD_M2,group,exporter
0,2015-01-04,Abu Dhabi,69.65999,20.746751,16.884515,17.004334,Exporter,1.0
1,2015-01-11,Abu Dhabi,71.70000,20.750363,16.079664,18.411733,Exporter,1.0
2,2015-01-18,Abu Dhabi,68.68999,20.746132,16.042590,16.450459,Exporter,1.0
3,2015-01-25,Abu Dhabi,68.67999,20.745758,16.027682,16.176663,Exporter,1.0
4,2015-02-01,Abu Dhabi,68.70000,20.738952,15.573514,15.963429,Exporter,1.0
...,...,...,...,...,...,...,...,...
8869,2024-12-01,Turkey,251.12000,-0.384709,-0.380030,-0.384708,Control,0.0
8870,2024-12-08,Turkey,241.90000,-0.360856,-0.356184,-0.360855,Control,0.0
8871,2024-12-15,Turkey,247.11000,-0.364650,-0.360640,-0.364649,Control,0.0
8872,2024-12-22,Turkey,256.52000,-0.367613,-0.363341,-0.367612,Control,0.0


In [8]:
# Cell 2: Load controls and merge onto panel

# ── Load macro controls ───────────────────────────────────────
macro = pd.read_csv(
    '../data/processed/Macroeconomic_variables/macro_risk_variables.csv',
    sep=',', dayfirst=True, parse_dates=['Date'], index_col='Date'
)
vix = pd.read_csv(
    '../data/processed/Macroeconomic_variables/VIXCLS.csv',
    parse_dates=['Date'], index_col='Date'
)
ovx = pd.read_csv(
    '../data/processed/Macroeconomic_variables/OVXCLS.csv',
    parse_dates=['date'], index_col='date'
)
gpr = pd.read_csv(
    '../data/processed/Macroeconomic_variables/geopolitical_risk_index_daily.csv',
    sep=';', dayfirst=True, parse_dates=['date'], index_col='date',
    decimal=','
)
for col in gpr.columns:
    gpr[col] = pd.to_numeric(gpr[col], errors='coerce')

oil_prices  = pd.read_csv('../data/processed/Oil/oil_prices_datastream.csv',
                           parse_dates=['date'], index_col='date')
oil_futures = pd.read_csv('../data/processed/Oil/oil_futures.csv',
                           dayfirst=True, parse_dates=['date'], index_col='date')

# ── Build controls on daily index ────────────────────────────
macro['VIX']   = vix['VIXCLS']
macro['OVX']   = ovx['OVXCLS']
macro          = macro.join(gpr[['GPRD']], how='left')
macro['basis'] = oil_prices['Brent'] / oil_futures['Brent_12m']
controls_daily = macro[['VIX', 'OVX', 'DXY', 'UST10Y', 'GPRD', 'basis']].copy()

# ── Load FX rates ─────────────────────────────────────────────
fx = pd.read_csv('../data/processed/CCA/cca_newfx_rates.csv',
                 parse_dates=['date'])
fx = fx[fx['country'].isin(EXPORTERS + CONTROLS)]\
       [['date', 'country', 'fx_rate']].copy()
fx = fx.rename(columns={'date': DATE_COL, 'country': COUNTRY_COL})

# ── Anchor to panel dates ─────────────────────────────────────
cds_dates = pd.DatetimeIndex(panel[DATE_COL].unique())

# ── Reindex controls to panel dates ──────────────────────────
controls_weekly = controls_daily\
    .reindex(cds_dates, method='nearest',
             tolerance=pd.Timedelta('7 days'))\
    .ffill().bfill()\
    .reset_index()\
    .rename(columns={'Date': DATE_COL, 'index': DATE_COL})
controls_weekly[DATE_COL] = pd.to_datetime(controls_weekly[DATE_COL])

# ── Reindex FX per country to panel dates ────────────────────
fx_weekly = pd.merge_asof(
    panel[[DATE_COL, COUNTRY_COL]].sort_values(DATE_COL),
    fx.sort_values(DATE_COL),
    on=DATE_COL,
    by=COUNTRY_COL,
    direction='nearest',
    tolerance=pd.Timedelta('7 days')
)

# ── Merge controls onto panel ─────────────────────────────────
panel = pd.merge_asof(
    panel.sort_values(DATE_COL),
    controls_weekly.sort_values(DATE_COL),
    on=DATE_COL,
    direction='nearest',
    tolerance=pd.Timedelta('7 days')
)

panel = panel.merge(
    fx_weekly[[DATE_COL, COUNTRY_COL, 'fx_rate']],
    on=[DATE_COL, COUNTRY_COL],
    how='left'
)

panel = panel.sort_values([COUNTRY_COL, DATE_COL]).reset_index(drop=True)

print("Panel shape:", panel.shape)
print("Columns:", panel.columns.tolist())
print("Nulls:\n", panel.isnull().sum()[panel.isnull().sum() > 0])

C:\Users\JuanFranciscoPerez\AppData\Local\Temp\ipykernel_24640\3457119148.py:24: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  oil_prices  = pd.read_csv('../data/processed/Oil/oil_prices_datastream.csv',


Panel shape: (8248, 15)
Columns: ['date', 'country', 'cds_spread', 'DD_M0', 'DD_M1', 'DD_M2', 'group', 'exporter', 'VIX', 'OVX', 'DXY', 'UST10Y', 'GPRD', 'basis', 'fx_rate']
Nulls:
 Series([], dtype: int64)


In [9]:
# Cell 2b: log-log transformation
panel = panel[panel['DD_M0'] > 0].copy()  # drop 52 degenerate rows

for model in ['M0', 'M1', 'M2']:
    panel[f'log_DD_{model}'] = np.log(panel[f'DD_{model}'].clip(lower=1e-6))

panel['log_cds'] = np.log(panel[CDS_COL])
panel = panel[np.isfinite(panel['log_cds'])].copy()
print(f"Panel after cleaning: {panel.shape}")

Panel after cleaning: (8196, 19)


# 1. Inseparability Test: OVX and Futures Basis vs Global Risk Factors

In [10]:
# Use unique dates from panel — one row per date for time series regressions
ts = panel.drop_duplicates(subset=[DATE_COL]).set_index(DATE_COL)\
          [['VIX', 'OVX', 'DXY', 'UST10Y', 'GPRD', 'basis']].sort_index()

# ── OVX ~ Global factors (levels) ────────────────────────────
idx   = ts[['OVX', 'VIX', 'DXY', 'UST10Y', 'GPRD']].dropna().index
y_ovx = ts.loc[idx, 'OVX']
X_ovx = sm.add_constant(ts.loc[idx, ['VIX', 'DXY', 'UST10Y', 'GPRD']])
res_ovx = sm.OLS(y_ovx, X_ovx).fit(cov_type='HAC', cov_kwds={'maxlags': 1})

print("OVX ~ Global Factors (levels)")
print(f"R²: {res_ovx.rsquared:.4f}  |  Adj. R²: {res_ovx.rsquared_adj:.4f}")
print(f"\n{'Variable':<12} {'Beta':>10} {'p-value':>10}")
print("-" * 35)
for var in X_ovx.columns:
    print(f"{var:<12} {res_ovx.params[var]:>10.4f} {res_ovx.pvalues[var]:>10.4f}")

OVX ~ Global Factors (levels)
R²: 0.5903  |  Adj. R²: 0.5871

Variable           Beta    p-value
-----------------------------------
const         -101.0723     0.0000
VIX              1.4612     0.0000
DXY              1.3522     0.0000
UST10Y          -7.6407     0.0000
GPRD             0.0105     0.4536


In [11]:
idx2    = ts[['basis', 'VIX', 'DXY', 'UST10Y', 'GPRD']].dropna().index
y_basis = ts.loc[idx2, 'basis']
X_basis = sm.add_constant(ts.loc[idx2, ['VIX', 'DXY', 'UST10Y', 'GPRD']])
res_basis = sm.OLS(y_basis, X_basis).fit(cov_type='HAC', cov_kwds={'maxlags': 1})

print("\nFutures Basis ~ Global Factors (levels)")
print(f"R²: {res_basis.rsquared:.4f}  |  Adj. R²: {res_basis.rsquared_adj:.4f}")
print(f"\n{'Variable':<15} {'Beta':>10} {'p-value':>10}")
print("-" * 38)
for var in X_basis.columns:
    print(f"{var:<15} {res_basis.params[var]:>10.4f} {res_basis.pvalues[var]:>10.4f}")


Futures Basis ~ Global Factors (levels)
R²: 0.1893  |  Adj. R²: 0.1830

Variable              Beta    p-value
--------------------------------------
const               1.1301     0.0000
VIX                 0.0002     0.8920
DXY                -0.0025     0.0826
UST10Y              0.0409     0.0000
GPRD                0.0002     0.0890


# 2. Interaction Regression: Exporter Differential After Global Controls
Following Jeanneret (2015), we regress observed CDS spreads on model-implied
distance-to-distress in levels with country fixed effects and time-clustered
standard errors. The interaction term DD × Exporter tests whether the
DD-CDS relationship is significantly stronger for oil-exporting sovereigns.

In [17]:
# Interaction regression: log(CDS) ~ log(DD) + log(DD)×Exporter + global controls
from linearmodels.panel import PanelOLS

interaction_results = []

for model_name in ['M0', 'M1', 'M2']:
    log_dd_col = f'log_DD_{model_name}'

    gdf = panel[[DATE_COL, COUNTRY_COL, 'log_cds', log_dd_col]]\
          .dropna().copy()
    gdf = gdf[np.isfinite(gdf[log_dd_col])]
    gdf = gdf.set_index([COUNTRY_COL, DATE_COL])

    y = gdf['log_cds']
    X = gdf[[log_dd_col]]

    res = PanelOLS(y, X, entity_effects=True).fit(
        cov_type='kernel', kernel='bartlett', bandwidth=4
    )

    interaction_results.append({
        'model':         model_name,
        'beta_log_dd':   res.params[log_dd_col],
        'p_log_dd':      res.pvalues[log_dd_col],
        'r2':            res.rsquared,
        'n_obs':         int(res.nobs),
    })

int_df = pd.DataFrame(interaction_results)
print(int_df.to_string(index=False))

model  beta_log_dd     p_log_dd       r2  n_obs
   M0    -0.217671 8.659740e-15 0.060609   8196
   M1    -0.236560 1.152546e-09 0.066719   8196
   M2    -0.217551 8.637535e-14 0.058575   8196


In [ ]:
# Interaction regression: log(CDS) ~ log(DD) + log(DD)×Exporter + global controls
from linearmodels.panel import PanelOLS

interaction_results = []

for model_name in ['M0', 'M1', 'M2']:
    log_dd_col = f'log_DD_{model_name}'

    gdf = panel[[DATE_COL, COUNTRY_COL, 'log_cds', log_dd_col,
                 'exporter', 'VIX', 'DXY', 'UST10Y', 'GPRD', 'fx_rate']]\
          .dropna().copy()
    gdf = gdf[np.isfinite(gdf[log_dd_col])]

    gdf['log_vix']   = np.log(gdf['VIX'])
    gdf['log_dxy']   = np.log(gdf['DXY'])
    gdf['log_ust']   = np.log(gdf['UST10Y'].clip(lower=0.001))
    gdf['log_gprd']  = np.log(gdf['GPRD'])
    gdf['log_fx']    = np.log(gdf['fx_rate'])
    gdf = gdf.set_index([COUNTRY_COL, DATE_COL])

    y = gdf['log_cds']
    X = gdf[[log_dd_col, 'dd_x_exp',
             'log_vix', 'log_dxy', 'log_ust', 'log_gprd', 'log_fx']]

    res = PanelOLS(y, X, entity_effects=False).fit(
        cov_type='kernel', kernel='bartlett', bandwidth=4
    )

    interaction_results.append({
        'model':         model_name,
        'beta_log_dd':   res.params[log_dd_col],
        'p_log_dd':      res.pvalues[log_dd_col],
        'beta_interact': res.params['dd_x_exp'],
        'se_interact':   res.std_errors['dd_x_exp'],
        'p_interact':    res.pvalues['dd_x_exp'],
        'beta_vix':      res.params['log_vix'],
        'p_vix':         res.pvalues['log_vix'],
        'beta_dxy':      res.params['log_dxy'],
        'p_dxy':         res.pvalues['log_dxy'],
        'beta_ust':      res.params['log_ust'],
        'p_ust':         res.pvalues['log_ust'],
        'beta_gpr':      res.params['log_gprd'],
        'p_gpr':         res.pvalues['log_gprd'],
        'beta_fx':       res.params['log_fx'],
        'p_fx':          res.pvalues['log_fx'],
        'r2':            res.rsquared,
        'n_obs':         int(res.nobs),
    })

int_df = pd.DataFrame(interaction_results)
print(int_df.to_string(index=False))

model  beta_log_dd  p_log_dd  beta_interact  se_interact  p_interact  beta_vix    p_vix  beta_dxy  p_dxy  beta_ust    p_ust  beta_gpr    p_gpr   beta_fx  p_fx       r2  n_obs
   M0    -0.752168       0.0       0.063890     0.013706    0.000003  0.076921 0.161077  1.260947    0.0 -0.049213 0.297875   0.04035 0.149879 -0.040225   0.0 0.981738   8196
   M1    -0.785285       0.0       0.066533     0.013867    0.000002  0.030477 0.606482  1.266969    0.0  0.022579 0.659924   0.04842 0.089867 -0.039977   0.0 0.981452   8196
   M2    -0.758108       0.0       0.061589     0.013449    0.000005  0.081148 0.155493  1.266658    0.0 -0.055681 0.247938   0.03586 0.208004 -0.040079   0.0 0.981723   8196


In [20]:
from scipy.optimize import minimize_scalar

def fit_inverse(x, y):
    """
    Grid search over delta, then OLS on 1/(x - delta).
    Returns beta0, beta1, delta, r2.
    """
    best_r2    = -np.inf
    best_result = None

    # delta must be strictly below min(x) so 1/(x-delta) is always positive
    delta_max = x.min() - 1e-4

    def neg_r2(delta):
        z = 1.0 / (x - delta)
        if not np.all(np.isfinite(z)):
            return 1.0  # bad
        X_ = np.column_stack([np.ones_like(z), z])
        b, *_ = np.linalg.lstsq(X_, y, rcond=None)
        y_hat = X_ @ b
        ss_res = np.sum((y - y_hat) ** 2)
        ss_tot = np.sum((y - y.mean()) ** 2)
        return ss_res / ss_tot if ss_tot > 0 else 1.0

    res = minimize_scalar(
        neg_r2,
        bounds=(x.min() - 20, delta_max),
        method='bounded',
        options={'xatol': 1e-6}
    )

    delta = res.x
    z     = 1.0 / (x - delta)
    X_    = np.column_stack([np.ones_like(z), z])
    b, *_ = np.linalg.lstsq(X_, y, rcond=None)
    y_hat = X_ @ b
    ss_res = np.sum((y - y_hat) ** 2)
    ss_tot = np.sum((y - y.mean()) ** 2)
    r2 = 1 - ss_res / ss_tot

    return b[0], b[1], delta, r2


nls_results = []

for model_name in ['M0', 'M1', 'M2']:
    dd_col = f'DD_{model_name}'

    for country in EXPORTERS + CONTROLS:
        d = panel[panel[COUNTRY_COL] == country]\
            .dropna(subset=[dd_col, CDS_COL]).copy()
        d = d[d[dd_col] > 0]

        if len(d) < 30:
            continue

        x = d[dd_col].values
        y = d[CDS_COL].values

        try:
            beta0, beta1, delta, r2 = fit_inverse(x, y)
            nls_results.append({
                'model':   model_name,
                'country': country,
                'group':   'Exporter' if country in EXPORTERS else 'Control',
                'beta0':   beta0,
                'beta1':   beta1,
                'delta':   delta,
                'r2':      r2,
                'n_obs':   len(d),
            })
        except Exception:
            nls_results.append({
                'model': model_name, 'country': country,
                'group': 'Exporter' if country in EXPORTERS else 'Control',
                'beta0': np.nan, 'beta1': np.nan, 'delta': np.nan,
                'r2': np.nan, 'n_obs': len(d),
            })

nls_df = pd.DataFrame(nls_results)

for model_name in ['M0', 'M1', 'M2']:
    print(f"\n{'='*60}")
    print(f"Model: {model_name}")
    print(f"{'='*60}")
    sub = nls_df[nls_df['model'] == model_name].sort_values(['group', 'country'])
    print(sub[['country', 'group', 'beta0', 'beta1', 'delta', 'r2', 'n_obs']]\
          .round(4).to_string(index=False))

print(f"\n{'='*60}")
print("Group summary")
print(f"{'='*60}")
print(nls_df.groupby(['model', 'group'])[['beta0', 'beta1', 'delta', 'r2']]\
      .mean().round(4).to_string())


Model: M0
     country    group     beta0      beta1    delta     r2  n_obs
       Chile  Control   55.6184    35.7071   1.1120 0.2569    522
       China  Control   60.3452    10.0061   6.7028 0.3453    522
   Indonesia  Control  -50.8044  3877.1551 -16.4741 0.0585    510
 Philippines  Control  113.0149  -875.7299 -14.5632 0.0432    516
South Africa  Control  189.4743    50.7828   2.4726 0.1660    522
 South Korea  Control   93.5509 -1258.5195 -13.8133 0.1365    519
    Thailand  Control  165.5263 -2657.1807 -12.9887 0.3056    522
      Turkey  Control  280.3554   124.1843   0.4523 0.4231    470
   Abu Dhabi Exporter   64.1604   -21.2981   4.4772 0.1217    514
      Brazil Exporter   64.3345   952.0415   0.1362 0.2348    522
    Colombia Exporter -334.7106  6916.4018  -9.5273 0.3748    522
       Egypt Exporter -311.5967 21288.7924 -19.2911 0.1428    522
    Malaysia Exporter   56.0739    75.6440   5.0063 0.2394    513
      Mexico Exporter -111.4311  5208.0175 -16.6810 0.1680    521

In [133]:
interaction_results = []

for model_name, path in PATHS.items():
    df = pd.read_csv(path, parse_dates=[DATE_COL],
                     usecols=[DATE_COL, COUNTRY_COL, DD_COL])
    df = df.sort_values([COUNTRY_COL, DATE_COL])
    df = df.dropna(subset=[DD_COL])

    # Compute model-implied CDS then log it
    df['cds_implied']     = dd_to_implied_cds(df[DD_COL], df[COUNTRY_COL])
    df['log_cds_implied'] = np.log(df['cds_implied'])
    df = df[np.isfinite(df['log_cds_implied'])]

    cds = pd.read_csv(PATHS['M0'], parse_dates=[DATE_COL],
                      usecols=[DATE_COL, COUNTRY_COL, CDS_COL, 'fx_rate'])
    cds = cds.rename(columns={'fx_rate': 'fx_rate_model'})
    cds['log_cds_obs'] = np.log(cds[CDS_COL])
    cds['log_fx']      = np.log(cds['fx_rate_model'])
    cds = cds[np.isfinite(cds['log_cds_obs']) & np.isfinite(cds['log_fx'])]

    df = df.merge(cds[[DATE_COL, COUNTRY_COL,
                        'log_cds_obs', 'log_fx']],
                  on=[DATE_COL, COUNTRY_COL], how='inner')
    df = df.dropna(subset=['log_cds_obs', 'log_fx'])

    # Log global controls
    global_levels = controls_weekly[['VIX', 'OVX', 'DXY', 'UST10Y', 'GPRD']]\
                    .reset_index()\
                    .rename(columns={'date': DATE_COL}).copy()
    global_levels['log_vix']  = np.log(global_levels['VIX'])
    global_levels['log_dxy']  = np.log(global_levels['DXY'])
    global_levels['log_ust']  = np.log(global_levels['UST10Y'].clip(lower=0.001))
    global_levels['log_gprd'] = np.log(global_levels['GPRD'])
    global_levels['log_ovx']  = np.log(global_levels['OVX'])

    df = pd.merge_asof(
        df.sort_values(DATE_COL),
        global_levels[['date', 'log_vix', 'log_dxy',
                        'log_ust', 'log_gprd', 'log_ovx']]\
                     .rename(columns={'date': DATE_COL})\
                     .sort_values(DATE_COL),
        on=DATE_COL,
        direction='nearest',
        tolerance=pd.Timedelta('7 days')
    )
    df = df.dropna(subset=['log_vix', 'log_dxy',
                            'log_ust', 'log_gprd', 'log_fx'])

    gdf = df[df[COUNTRY_COL].isin(EXPORTERS + CONTROLS)].copy()
    gdf['exporter']          = gdf[COUNTRY_COL].isin(EXPORTERS).astype(float)
    gdf['log_imp_x_exp']     = gdf['log_cds_implied'] * gdf['exporter']
    gdf = gdf.set_index([COUNTRY_COL, DATE_COL])

    y = gdf['log_cds_obs']
    X = gdf[['log_cds_implied', 'log_imp_x_exp',
             'log_vix', 'log_dxy', 'log_ust',
             'log_gprd', 'log_fx']]

    res = PanelOLS(y, X, entity_effects=False).fit(
        cov_type='kernel',
        kernel='bartlett',
        bandwidth=1
    )

    interaction_results.append({
        'model':         model_name,
        'beta_cds_imp':  res.params['log_cds_implied'],
        'p_cds_imp':     res.pvalues['log_cds_implied'],
        'beta_interact': res.params['log_imp_x_exp'],
        'se_interact':   res.std_errors['log_imp_x_exp'],
        'p_interact':    res.pvalues['log_imp_x_exp'],
        'beta_vix':      res.params['log_vix'],
        'p_vix':         res.pvalues['log_vix'],
        'beta_dxy':      res.params['log_dxy'],
        'p_dxy':         res.pvalues['log_dxy'],
        'beta_ust':      res.params['log_ust'],
        'p_ust':         res.pvalues['log_ust'],
        'beta_gpr':      res.params['log_gprd'],
        'p_gpr':         res.pvalues['log_gprd'],
        'beta_fx':       res.params['log_fx'],
        'p_fx':          res.pvalues['log_fx'],
        'r2':            res.rsquared,
        'n_obs':         int(res.nobs),
    })

int_df = pd.DataFrame(interaction_results)
print(int_df.to_string(index=False))

model  beta_cds_imp  p_cds_imp  beta_interact  se_interact   p_interact  beta_vix    p_vix  beta_dxy  p_dxy  beta_ust        p_ust  beta_gpr    p_gpr   beta_fx  p_fx       r2  n_obs
   M0      0.129655        0.0      -0.029596     0.003099 0.000000e+00  0.078532 0.085632  1.134520    0.0 -0.035157 2.812720e-01 -0.009223 0.743612 -0.029120   0.0 0.979357   8011
   M1      0.100522        0.0      -0.027195     0.003894 3.091527e-12 -0.105221 0.048381  1.075320    0.0  0.212134 1.598230e-07  0.046100 0.184333 -0.035313   0.0 0.978492   8075
   M2      0.088789        0.0      -0.031792     0.003453 0.000000e+00 -0.122161 0.007689  1.163499    0.0 -0.022029 4.773347e-01 -0.045113 0.122053 -0.059725   0.0 0.977075   8080


In [134]:
interaction_results_simple = []

for model_name, path in PATHS.items():
    df = pd.read_csv(path, parse_dates=[DATE_COL],
                     usecols=[DATE_COL, COUNTRY_COL, DD_COL])
    df = df.sort_values([COUNTRY_COL, DATE_COL])
    df = df.dropna(subset=[DD_COL])

    df['cds_implied']     = dd_to_implied_cds(df[DD_COL], df[COUNTRY_COL])
    df['log_cds_implied'] = np.log(df['cds_implied'])
    df = df[np.isfinite(df['log_cds_implied'])]

    cds = pd.read_csv(PATHS['M0'], parse_dates=[DATE_COL],
                      usecols=[DATE_COL, COUNTRY_COL, CDS_COL])
    cds['log_cds_obs'] = np.log(cds[CDS_COL])
    cds = cds[np.isfinite(cds['log_cds_obs'])]

    df = df.merge(cds[[DATE_COL, COUNTRY_COL, 'log_cds_obs']],
                  on=[DATE_COL, COUNTRY_COL], how='inner')
    df = df.dropna(subset=['log_cds_obs'])

    gdf = df[df[COUNTRY_COL].isin(EXPORTERS + CONTROLS)].copy()
    gdf['const']         = 1.0
    gdf['exporter']      = gdf[COUNTRY_COL].isin(EXPORTERS).astype(float)
    gdf['log_imp_x_exp'] = gdf['log_cds_implied'] * gdf['exporter']
    gdf = gdf.set_index([COUNTRY_COL, DATE_COL])

    y = gdf['log_cds_obs']
    X = gdf[['const', 'log_cds_implied', 'log_imp_x_exp']]


    res = PanelOLS(y, X, entity_effects=False, time_effects=True).fit(
        cov_type='kernel',
        kernel='bartlett',
        bandwidth=4
    )

    interaction_results_simple.append({
        'model':         model_name,
        'beta_cds_imp':  res.params['log_cds_implied'],
        'p_cds_imp':     res.pvalues['log_cds_implied'],
        'beta_interact': res.params['log_imp_x_exp'],
        'se_interact':   res.std_errors['log_imp_x_exp'],
        'p_interact':    res.pvalues['log_imp_x_exp'],
        'r2':            res.rsquared,
        'n_obs':         int(res.nobs),
    })

simple_df = pd.DataFrame(interaction_results_simple)
print(simple_df.to_string(index=False))

model  beta_cds_imp  p_cds_imp  beta_interact  se_interact   p_interact       r2  n_obs
   M0      0.137011        0.0      -0.046142     0.004426 0.000000e+00 0.270124   8283
   M1      0.132522        0.0      -0.037933     0.006256 1.394094e-09 0.278637   8347
   M2      0.080430        0.0      -0.027984     0.005560 4.939123e-07 0.144878   8352


In [135]:
group_results = []

for model_name, path in PATHS.items():
    df = pd.read_csv(path, parse_dates=[DATE_COL],
                     usecols=[DATE_COL, COUNTRY_COL, DD_COL])
    df = df.sort_values([COUNTRY_COL, DATE_COL])
    df = df.dropna(subset=[DD_COL])

    # Compute model-implied CDS then log
    df['cds_implied']     = dd_to_implied_cds(df[DD_COL], df[COUNTRY_COL])
    df['log_cds_implied'] = np.log(df['cds_implied'])
    df = df[np.isfinite(df['log_cds_implied'])]

    cds = pd.read_csv(PATHS['M0'], parse_dates=[DATE_COL],
                      usecols=[DATE_COL, COUNTRY_COL, CDS_COL, 'fx_rate'])
    cds = cds.rename(columns={'fx_rate': 'fx_rate_model'})
    cds['log_cds_obs'] = np.log(cds[CDS_COL])
    cds['log_fx']      = np.log(cds['fx_rate_model'])
    cds = cds[np.isfinite(cds['log_cds_obs']) & np.isfinite(cds['log_fx'])]

    df = df.merge(cds[[DATE_COL, COUNTRY_COL,
                        'log_cds_obs', 'log_fx']],
                  on=[DATE_COL, COUNTRY_COL], how='inner')
    df = df.dropna(subset=['log_cds_obs', 'log_fx'])

    # Log global controls
    global_levels = controls_weekly[['VIX', 'DXY', 'UST10Y', 'GPRD']]\
                    .reset_index()\
                    .rename(columns={'date': DATE_COL}).copy()
    global_levels['log_vix']  = np.log(global_levels['VIX'])
    global_levels['log_dxy']  = np.log(global_levels['DXY'])
    global_levels['log_ust']  = np.log(global_levels['UST10Y'].clip(lower=0.001))
    global_levels['log_gprd'] = np.log(global_levels['GPRD'])

    df = pd.merge_asof(
        df.sort_values(DATE_COL),
        global_levels[['date', 'log_vix', 'log_dxy',
                        'log_ust', 'log_gprd']]\
                     .rename(columns={'date': DATE_COL})\
                     .sort_values(DATE_COL),
        on=DATE_COL,
        direction='nearest',
        tolerance=pd.Timedelta('7 days')
    )
    df = df.dropna(subset=['log_vix', 'log_dxy', 'log_ust',
                            'log_gprd', 'log_fx'])

    for group_name, group_countries in [('Exporters', EXPORTERS),
                                         ('Controls',  CONTROLS)]:
        gdf = df[df[COUNTRY_COL].isin(group_countries)].copy()
        gdf['const'] = 1.0
        gdf = gdf.set_index([COUNTRY_COL, DATE_COL])

        y = gdf['log_cds_obs']
        X = gdf[['const', 'log_cds_implied']]

        res = PanelOLS(y, X, entity_effects=False).fit(
            cov_type='kernel',
            kernel='bartlett',
            bandwidth=2
        )

        group_results.append({
            'model':        model_name,
            'group':        group_name,
            'beta_cds_imp': res.params['log_cds_implied'],
            'se_cds_imp':   res.std_errors['log_cds_implied'],
            'p_cds_imp':    res.pvalues['log_cds_implied'],
            'r2':           res.rsquared,
            'n_obs':        int(res.nobs),
        })

group_df = pd.DataFrame(group_results)
print(group_df.to_string(index=False))

# ── Wald test: exporter beta vs control beta per model ────────────────
print("\n" + "=" * 60)
print("Wald test: Exporter vs Control log implied CDS coefficient")
print("H0: beta_exporters = beta_controls")
print("=" * 60)

for model_name in ['M0', 'M1', 'M2']:
    row_exp  = group_df[(group_df['model']==model_name) &
                        (group_df['group']=='Exporters')]
    row_ctrl = group_df[(group_df['model']==model_name) &
                        (group_df['group']=='Controls')]

    b_exp   = row_exp['beta_cds_imp'].values[0]
    b_ctrl  = row_ctrl['beta_cds_imp'].values[0]
    se_exp  = row_exp['se_cds_imp'].values[0]
    se_ctrl = row_ctrl['se_cds_imp'].values[0]

    z = (b_exp - b_ctrl) / np.sqrt(se_exp**2 + se_ctrl**2)
    p = 2 * (1 - stats.norm.cdf(abs(z)))

    print(f"\n{model_name}:")
    print(f"  Exporters beta: {b_exp:.4f} (SE={se_exp:.4f})")
    print(f"  Controls  beta: {b_ctrl:.4f} (SE={se_ctrl:.4f})")
    print(f"  Difference:     {b_exp - b_ctrl:.4f}")
    print(f"  Z-statistic:    {z:.4f}")
    print(f"  P-value:        {p:.6f}")

# ── Wald test: difference of differences ─────────────────────────────
print("\n" + "=" * 60)
print("Wald test: Difference of differences (M1/M2 vs M0)")
print("=" * 60)

for alt_model in ['M1', 'M2']:
    diff_m0  = (group_df[(group_df['model']=='M0') & (group_df['group']=='Exporters')]['beta_cds_imp'].values[0] -
                group_df[(group_df['model']=='M0') & (group_df['group']=='Controls')]['beta_cds_imp'].values[0])
    diff_alt = (group_df[(group_df['model']==alt_model) & (group_df['group']=='Exporters')]['beta_cds_imp'].values[0] -
                group_df[(group_df['model']==alt_model) & (group_df['group']=='Controls')]['beta_cds_imp'].values[0])

    se_m0  = np.sqrt(
        group_df[(group_df['model']=='M0') & (group_df['group']=='Exporters')]['se_cds_imp'].values[0]**2 +
        group_df[(group_df['model']=='M0') & (group_df['group']=='Controls')]['se_cds_imp'].values[0]**2
    )
    se_alt = np.sqrt(
        group_df[(group_df['model']==alt_model) & (group_df['group']=='Exporters')]['se_cds_imp'].values[0]**2 +
        group_df[(group_df['model']==alt_model) & (group_df['group']=='Controls')]['se_cds_imp'].values[0]**2
    )

    z = (diff_alt - diff_m0) / np.sqrt(se_m0**2 + se_alt**2)
    p = 2 * (1 - stats.norm.cdf(abs(z)))

    print(f"\nM0 vs {alt_model}:")
    print(f"  M0  gap: {diff_m0:.4f}")
    print(f"  {alt_model} gap: {diff_alt:.4f}")
    print(f"  DiD:     {diff_alt - diff_m0:.4f}")
    print(f"  Z:       {z:.4f}")
    print(f"  P-value: {p:.6f}")

model     group  beta_cds_imp  se_cds_imp    p_cds_imp       r2  n_obs
   M0 Exporters      0.130177    0.008489 0.000000e+00 0.226072   3988
   M0  Controls      0.112372    0.003762 0.000000e+00 0.254832   4023
   M1 Exporters      0.061793    0.008318 1.332268e-13 0.115256   4037
   M1  Controls      0.082407    0.007217 0.000000e+00 0.234490   4038
   M2 Exporters      0.035268    0.003801 0.000000e+00 0.031814   4040
   M2  Controls      0.082561    0.003778 0.000000e+00 0.252914   4040

Wald test: Exporter vs Control log implied CDS coefficient
H0: beta_exporters = beta_controls

M0:
  Exporters beta: 0.1302 (SE=0.0085)
  Controls  beta: 0.1124 (SE=0.0038)
  Difference:     0.0178
  Z-statistic:    1.9175
  P-value:        0.055174

M1:
  Exporters beta: 0.0618 (SE=0.0083)
  Controls  beta: 0.0824 (SE=0.0072)
  Difference:     -0.0206
  Z-statistic:    -1.8719
  P-value:        0.061220

M2:
  Exporters beta: 0.0353 (SE=0.0038)
  Controls  beta: 0.0826 (SE=0.0038)
  Difference:  

In [136]:
from linearmodels.panel import PooledOLS
import scipy.stats as stats

# ── Univariate: observed CDS ~ model-implied CDS ──────────────
rmse_results = []

for model_name, path in PATHS.items():
    df = pd.read_csv(path, parse_dates=[DATE_COL],
                     usecols=[DATE_COL, COUNTRY_COL, DD_COL])
    df = df.sort_values([COUNTRY_COL, DATE_COL])
    df = df.dropna(subset=[DD_COL])

    df['cds_implied'] = dd_to_implied_cds(df[DD_COL], df[COUNTRY_COL])

    cds = pd.read_csv(PATHS['M0'], parse_dates=[DATE_COL],
                      usecols=[DATE_COL, COUNTRY_COL, CDS_COL])
    df = df.merge(cds, on=[DATE_COL, COUNTRY_COL], how='inner')
    df = df.dropna(subset=[CDS_COL, 'cds_implied'])
    df = df[np.isfinite(df['cds_implied'])]

    for country in EXPORTERS + CONTROLS:
        cdf = df[df[COUNTRY_COL] == country].copy()
        if len(cdf) < 10:
            continue

        # Univariate OLS: observed CDS ~ implied CDS
        X = sm.add_constant(cdf['cds_implied'])
        y = cdf[CDS_COL]
        ols = sm.OLS(y, X).fit()

        fitted  = ols.fittedvalues
        resid   = y - fitted
        rmse    = np.sqrt((resid**2).mean())
        mae     = resid.abs().mean()
        beta    = ols.params['cds_implied']
        r2      = ols.rsquared

        rmse_results.append({
            'model':   model_name,
            'country': country,
            'group':   'Exporter' if country in EXPORTERS else 'Control',
            'beta':    beta,
            'r2':      r2,
            'rmse':    rmse,
            'mae':     mae,
            'n_obs':   len(cdf),
        })

rmse_df = pd.DataFrame(rmse_results)

# ── Per country table ─────────────────────────────────────────
for model_name in ['M0', 'M1', 'M2']:
    print(f"\n{'='*60}")
    print(f"Model: {model_name}")
    print(f"{'='*60}")
    subset = rmse_df[rmse_df['model'] == model_name]\
             .sort_values(['group', 'country'])
    print(subset[['country', 'group', 'beta', 'r2',
                  'rmse', 'mae']].to_string(index=False))

# ── Group summary ─────────────────────────────────────────────
print(f"\n{'='*60}")
print("Group RMSE summary")
print(f"{'='*60}")
summary = rmse_df.groupby(['model', 'group'])\
                 .agg(mean_beta=('beta', 'mean'),
                      mean_r2  =('r2',   'mean'),
                      mean_rmse=('rmse', 'mean'),
                      mean_mae =('mae',  'mean'))\
                 .round(4)
print(summary.to_string())

# ── Wald test: is RMSE lower for M1/M2 vs M0? ────────────────
print(f"\n{'='*60}")
print("RMSE improvement: M1 and M2 vs M0")
print(f"{'='*60}")

for alt_model in ['M1', 'M2']:
    for group_name in ['Exporter', 'Control']:
        rmse_m0  = rmse_df[(rmse_df['model']=='M0') &
                           (rmse_df['group']==group_name)]['rmse'].values
        rmse_alt = rmse_df[(rmse_df['model']==alt_model) &
                           (rmse_df['group']==group_name)]['rmse'].values

        # Paired t-test: is RMSE reduction significant?
        diff = rmse_m0 - rmse_alt  # positive = improvement
        t, p = stats.ttest_1samp(diff, 0)

        print(f"\nM0 vs {alt_model} | {group_name}:")
        print(f"  M0   mean RMSE: {rmse_m0.mean():.4f}")
        print(f"  {alt_model}   mean RMSE: {rmse_alt.mean():.4f}")
        print(f"  Mean reduction: {diff.mean():.4f}")
        print(f"  t-statistic:    {t:.4f}")
        print(f"  P-value:        {p:.6f}")


Model: M0
     country    group         beta            r2       rmse        mae
       Chile  Control     0.962622  2.650005e-01  24.285965  19.528009
       China  Control 57287.881061 -2.220446e-16  25.756514  20.615285
   Indonesia  Control    -2.087388  6.000261e-06  45.484905  36.563893
 Philippines  Control 64851.142556  0.000000e+00  24.990527  20.777277
South Africa  Control    49.245565  1.591038e-01  48.551318  37.689426
 South Korea  Control 34131.065448  0.000000e+00  14.297172  11.923408
    Thailand  Control 50250.360186 -2.220446e-16  31.502401  24.306349
      Turkey  Control    -0.015187  1.515042e-03 151.034544 121.243450
   Abu Dhabi Exporter 48196.071187  3.330669e-16  18.784696  13.996763
      Brazil Exporter 12300.456612  7.904809e-02  74.181910  57.200466
    Colombia Exporter    31.455919  1.990007e-01  55.560638  46.663190
       Egypt Exporter    -0.364874  9.926608e-03 354.832140 258.790172
    Malaysia Exporter 69888.082107  1.110223e-16  42.142435  34.63

In [137]:
error_results = []

for alt_model in ['M1', 'M2']:

    # ── Load M0 ───────────────────────────────────────────────
    df_m0 = pd.read_csv(PATHS['M0'], parse_dates=[DATE_COL],
                        usecols=[DATE_COL, COUNTRY_COL, DD_COL, CDS_COL])
    df_m0['cds_implied'] = dd_to_implied_cds(df_m0[DD_COL], df_m0[COUNTRY_COL])
    df_m0['log_imp']     = np.log(df_m0['cds_implied'])
    df_m0['log_obs']     = np.log(df_m0[CDS_COL])
    df_m0['sq_error']    = (df_m0['log_imp'] - df_m0['log_obs'])**2
    df_m0['model_ext']   = 0.0
    df_m0 = df_m0[np.isfinite(df_m0['log_imp']) &
                  np.isfinite(df_m0['log_obs'])]

    # ── Load MX ───────────────────────────────────────────────
    df_mx = pd.read_csv(PATHS[alt_model], parse_dates=[DATE_COL],
                        usecols=[DATE_COL, COUNTRY_COL, DD_COL])
    df_mx = df_mx.merge(
        df_m0[[DATE_COL, COUNTRY_COL, CDS_COL, 'log_obs']],
        on=[DATE_COL, COUNTRY_COL], how='inner'
    )
    df_mx['cds_implied'] = dd_to_implied_cds(df_mx[DD_COL], df_mx[COUNTRY_COL])
    df_mx['log_imp']     = np.log(df_mx['cds_implied'])
    df_mx['sq_error']    = (df_mx['log_imp'] - df_mx['log_obs'])**2
    df_mx['model_ext']   = 1.0
    df_mx = df_mx[np.isfinite(df_mx['log_imp']) &
                  np.isfinite(df_mx['log_obs'])]

    # ── Stack ─────────────────────────────────────────────────
    keep    = [DATE_COL, COUNTRY_COL, 'sq_error', 'model_ext']
    stacked = pd.concat([df_m0[keep], df_mx[keep]], ignore_index=True)
    stacked = stacked[stacked[COUNTRY_COL].isin(EXPORTERS + CONTROLS)].copy()
    stacked = stacked.reset_index(drop=True)

    stacked['exporter']  = stacked[COUNTRY_COL].isin(EXPORTERS).astype(float)
    stacked['ext_x_exp'] = stacked['model_ext'] * stacked['exporter']
    stacked['const']     = 1.0

    # ── Time fixed effects as float dummies ───────────────────
    time_dummies = pd.get_dummies(stacked[DATE_COL],
                                  prefix='t',
                                  drop_first=True).astype(float)
    time_dummies.index = stacked.index

    # ── Build X and y ─────────────────────────────────────────
    fe_cols = time_dummies.columns.tolist()
    X = pd.concat([stacked[['const', 'model_ext',
                             'exporter', 'ext_x_exp']],
                   time_dummies], axis=1).astype(float)
    y = stacked['sq_error'].astype(float)

    groups = stacked[COUNTRY_COL].values

    ols = sm.OLS(y, X).fit(
        cov_type='cluster'    )

    b_ext   = ols.params['model_ext']
    b_exp   = ols.params['exporter']
    b_delta = ols.params['ext_x_exp']
    se_delta= ols.bse['ext_x_exp']
    p_delta = ols.pvalues['ext_x_exp']
    p_ext   = ols.pvalues['model_ext']
    p_exp   = ols.pvalues['exporter']

    error_results.append({
        'alt_model': alt_model,
        'beta_ext':  b_ext,
        'p_ext':     p_ext,
        'beta_exp':  b_exp,
        'p_exp':     p_exp,
        'delta':     b_delta,
        'se_delta':  se_delta,
        'p_delta':   p_delta,
        'r2':        ols.rsquared,
        'n_obs':     int(ols.nobs),
    })

    print(f"\n{'='*60}")
    print(f"Squared error DiD: M0 vs {alt_model}")
    print(f"{'='*60}")
    print(f"  β  Extension:       {b_ext:.6f}  (p={p_ext:.4f})")
    print(f"  γ  Exporter:        {b_exp:.6f}  (p={p_exp:.4f})")
    print(f"  δ  Ext×Exp (H3):    {b_delta:.6f}  (p={p_delta:.4f})")
    print(f"  R²: {ols.rsquared:.4f}  |  N: {int(ols.nobs)}")
    print(f"\n  Negative δ = {alt_model} reduces errors MORE for exporters.")

err_df = pd.DataFrame(error_results)
print(f"\n{'='*60}")
print("Summary")
print(f"{'='*60}")
print(err_df[['alt_model', 'beta_ext', 'p_ext',
              'beta_exp', 'p_exp',
              'delta', 'se_delta', 'p_delta']].to_string(index=False))

KeyError: 'groups'